# Stage 11 — track reconciliation

Thin orchestration for final hard-gated, merge- and lineage-aware identity repair.

In [ ]:
from importlib import import_module

import numpy as np

from src.api import run_track_reconciliation
from src.io import (
    PipelinePaths,
    load_processed_dataset_inputs,
    load_stage7_outputs,
    load_stage8_outputs,
    load_stage10_outputs,
    save_track_reconciliation_result,
)

TrackReconciliationConfig = import_module(
    "src.11_track_reconciliation.step01_config"
).TrackReconciliationConfig

SAMPLE_ID = "44b6_0113de3b"
paths = PipelinePaths.discover()

In [ ]:
processed = load_processed_dataset_inputs(SAMPLE_ID, paths=paths)
stage7 = load_stage7_outputs(paths=paths)
stage8 = load_stage8_outputs(paths=paths)
stage10 = load_stage10_outputs(paths=paths)
spatial_shape_zyx = tuple(
    int(value) for value in np.load(
        processed.segmentation_files[0], mmap_mode="r"
    ).shape
)

In [ ]:
config = TrackReconciliationConfig(policy="submission")
result = run_track_reconciliation(
    stage8.tracks,
    list(processed.time_frames),
    segmentation_events=stage8.segmentation_events,
    division_events=stage10.division_events,
    lineage_edges=stage10.lineage_edges,
    track_lineage=stage10.track_lineage,
    protected_tracks=stage10.protected_tracks,
    global_motion=stage7.global_motion,
    association_events=stage7.association_events,
    association_candidates=stage7.association_candidates,
    spatial_shape_zyx=spatial_shape_zyx,
    sample_id=SAMPLE_ID,
    config=config,
)
save_track_reconciliation_result(result, paths.stage11_reconciliation)

In [ ]:
print("Input tracks:", result.summary["input_tracks"])
print("Final tracks:", result.summary["final_tracks"])
print("Candidate edges:", result.summary["candidate_edges"])
print("Conservative assignments:", result.summary["conservative_assignments"])
print("Forced assignments:", result.summary["forced_assignments"])
print("Unresolved eligible endings:", result.summary["unresolved_eligible_endings"])

In [ ]:
display(result.continuation_decisions["decision"].value_counts(dropna=False))
display(result.unresolved_endings["reason"].value_counts(dropna=False))